<a href="https://colab.research.google.com/github/hanghae-plus-AI/AI-1-gwkcareer/blob/main/week5/Chapter3_2_%EA%B8%B0%EB%B3%B8%EA%B3%BC%EC%A0%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [5주차 기본과제] 뉴스 기사 분류를 Gemma로 구현하기

이번 과제에서는 이전 주차 과제에서 활용했던 `fancyzhx/ag_news` 문제를 zero-shot classification으로 푸시면 됩니다. 아래 사항들에 유의하시면 될 것 같습니다.

- [ ]  Label들을 올바르게 text화 하여 넘겨주셔야 합니다.
- [ ]  `test` split data 50개에 대한 정확도 계산 코드 및 출력이 남아있어야 합니다.

이외에는 Gemma-2B 모델의 logit 계산 능력을 활용한다는 부분 빼고는 제약이 없습니다.

In [ ]:
!pip install datasets

INFO: pip is looking at multiple versions of multiprocess to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.6/471.6 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 16.1 MB/s eta 0:00:00


## 준비

1) huggingface 에서 token 생성

2) Gemma license 동의

In [ ]:
from huggingface_hub import login
from google.colab import userdata

login(userdata.get('HF_TOKEN_NEW')) # 파일로 숨기기
#login('') # 토큰 직접 넣기

The token has not been saved to the git credentials helper. Pass `add_to_git_credential=True` in this function directly or `--add-to-git-credential` if using via `huggingface-cli` if you want to set the git credential as well.
Token is valid (permission: write).
Your token has been saved to /root/.cache/huggingface/token
Login successful


## tokenizer와 Gemma-2B 모델 호출

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2b")
model = AutoModelForCausalLM.from_pretrained("google/gemma-2b", device_map="auto")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/33.6k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/4.24M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.5M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/636 [00:00<?, ?B/s]

...

model-00001-of-00002.safetensors:   0%|          | 0.00/4.95G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/67.1M [00:00<?, ?B/s]

`config.hidden_act` is ignored, you should use `config.hidden_activation` instead.
Gemma's activation function will be set to `gelu_pytorch_tanh`. Please, use
`config.hidden_activation` if you want to override this behaviour.
See https://github.com/huggingface/transformers/pull/29402 for more details.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/137 [00:00<?, ?B/s]

## Dataset 준비
fancyzhx/ag_news dataset을 load_dataset 함수로 다운

In [ ]:
from datasets import load_dataset
dataset = load_dataset("fancyzhx/ag_news")
dataset

# 라벨 이름 확인/저장
for label in dataset['train'].features['label'].names:
    print(label)
label_names = dataset['train'].features['label'].names


README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/18.6M [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/1.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

World
Sports
Business
Sci/Tech


In [ ]:
# 샘플 데이터 가져오기
input_text = dataset['train'][0]['text']

# 입력 텍스트를 토크나이징하고 텐서로 변환 (GPU 사용)
input_ids = tokenizer(input_text, return_tensors="pt").to("cuda")

# 모델에 입력하여 예측 생성
outputs = model.generate(**input_ids, max_new_tokens=50)

# 예측 결과 디코딩하여 출력
result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again. The number of bearish bets on the S&P 500 index has fallen to its lowest level in more than a year, according to a Reuters poll of 100 hedge funds. The number of short positions in the S&P


In [ ]:
# 1. 데이터 전처리 및 토크나이징
def preprocess_function(data):
    return tokenizer(data["text"], truncation=True)

ag_news_tokenized = dataset.map(preprocess_function, batched=True)

# 2. 데이터셋 분할
ag_news_split = ag_news_tokenized['train'].train_test_split(test_size=0.2)
trainset, ag_news_val = ag_news_split['train'], ag_news_split['test']
testset = ag_news_tokenized['test']

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


Map:   0%|          | 0/7600 [00:00<?, ? examples/s]

In [ ]:
# 3. 특정 샘플에 대해 토크나이저 적용 및 모델 입력
input_text = dataset['train'][0]['text']
input_ids = tokenizer(input_text, return_tensors="pt").to("cuda")

# 4. 토큰과 모델 출력 로짓 확인
tokens = input_ids['input_ids']
logits = model(**input_ids).logits

# 로짓 값 출력 해보기
for i in range(tokens.shape[-1]):
    token = tokens[0, i].item()  # 각 토큰의 ID를 가져옴
    print(logits[0, i, token])
    #print(f"Token: {token} ({tokenizer.decode([token])}), Logit: {logits[0, i, token].item()}")

tensor(-18.2746, device='cuda:0', grad_fn=<SelectBackward0>)
tensor(-19.1394, device='cuda:0', grad_fn=<SelectBackward0>)
tensor(-16.3865, device='cuda:0', grad_fn=<SelectBackward0>)
tensor(-28.4498, device='cuda:0', grad_fn=<SelectBackward0>)
tensor(-26.0919, device='cuda:0', grad_fn=<SelectBackward0>)
tensor(-20.0170, device='cuda:0', grad_fn=<SelectBackward0>)
tensor(-15.4359, device='cuda:0', grad_fn=<SelectBackward0>)
tensor(-14.5964, device='cuda:0', grad_fn=<SelectBackward0>)
tensor(-15.4441, device='cuda:0', grad_fn=<SelectBackward0>)
tensor(-16.3374, device='cuda:0', grad_fn=<SelectBackward0>)
tensor(-8.7537, device='cuda:0', grad_fn=<SelectBackward0>)
tensor(-17.0157, device='cuda:0', grad_fn=<SelectBackward0>)
tensor(-22.8966, device='cuda:0', grad_fn=<SelectBackward0>)
tensor(-14.9466, device='cuda:0', grad_fn=<SelectBackward0>)
tensor(-8.4882, device='cuda:0', grad_fn=<SelectBackward0>)
tensor(-20.8404, device='cuda:0', grad_fn=<SelectBackward0>)
tensor(-6.2513, device='cu

## zero-shot classification
(Logit은 높을 수록 token이 나올 확률이 높다)

In [ ]:
import torch

def zero_shot_classification(text, task_description, labels):  # text는 주어진 입력, task_description은 task에 대한 설명, labels은 class들을 text로 변환한 결과입니다.
    text_ids = tokenizer(task_description + text, return_tensors="pt").to("cuda")  # 먼저 task_description과 text를 이어붙인 후, tokenize합니다.
    probs = []

    for label in labels:  # 그 다음 각 text화된 label들을 tokenize하고 입력에 이어붙인 후, Gemma-2B에 넣어줍니다.
        label_ids = tokenizer(label, return_tensors="pt").to("cuda")
        n_label_tokens = label_ids['input_ids'].shape[-1] - 1  # text로 변환한 label의 token 수를 계산합니다.
        input_ids = {
            'input_ids': torch.concatenate([text_ids['input_ids'], label_ids['input_ids'][:, 1:]], axis=-1),  # concatenate 명령어를 통해 이어붙이는 모습입니다.
            'attention_mask': torch.concatenate([text_ids['attention_mask'], label_ids['attention_mask'][:, 1:]], axis=-1)
        }

        logits = model(**input_ids).logits  # Logit을 계산한 모습입니다.
        prob = 0
        n_total = input_ids['input_ids'].shape[-1]
        for i in range(n_label_tokens, 0, -1):  # 일반적으로 text로 변환한 label은 여러 token으로 이루어져있습니다. 이러한 label에 대한 logit은 구성하는 모든 token들의 logit들의 합으로 정의합니다.
            token = label_ids['input_ids'][0, i].item()
            prob += logits[0, n_total - i, token].item()
        probs.append(prob)

        del input_ids
        del logits
        torch.cuda.empty_cache()  # 위의 del과 empty_cache() 명령어를 통해 GPU를 제때 할당해제 해줍니다. 만약 GPU가 여유롭다면 지워주시는게 속도적으로 이득입니다.

    return probs

## 학습1
`- labels=["Answer: World.", "Answer: Sports.", "Answer: Business.", "Answer: Sci/Tech."]`

In [ ]:
n_corrects = 0
num_samples = 50  # 테스트할 샘플 수
incorrect_samples = []  # 잘못 예측한 샘플을 저장할 리스트

for i in tqdm(range(num_samples)):
    # `testset`에서 샘플 텍스트와 라벨을 가져오기
    text = testset[i]['text']
    true_label = testset[i]['label']

    # Zero-Shot Classification 수행
    probs = zero_shot_classification(
            text,
            "Given the following article, identify which of these categories it belongs to: ",
            labels=["Answer: World.", "Answer: Sports.", "Answer: Business.", "Answer: Sci/Tech."]
    )

    # 모델이 예측한 라벨 인덱스
    pred = np.argmax(np.array(probs))

    # 예측이 정답과 일치하는지 확인
    if pred == true_label:
        n_corrects += 1
    else:
        incorrect_samples.append((text, true_label, pred))  # 잘못 예측한 샘플 저장

print(n_corrects)

# 정확도 계산
accuracy = (n_corrects / num_samples) * 100
print(f"Accuracy: {accuracy:.2f}%")

# 잘못 예측한 샘플 출력
for sample in incorrect_samples:
    text, true_label, pred = sample
    print(f"\nText: {text}\nTrue Label: {true_label}, Predicted: {pred}")

100%|██████████| 50/50 [00:11<00:00,  4.32it/s]

25
Accuracy: 50.00%

Text: Ky. Company Wins Grant to Study Peptides (AP) AP - A company founded by a chemistry researcher at the University of Louisville won a grant to develop a method of producing better peptides, which are short chains of amino acids, the building blocks of proteins.
True Label: 3, Predicted: 2

Text: Prediction Unit Helps Forecast Wildfires (AP) AP - It's barely dawn when Mike Fitzpatrick starts his shift with a blur of colorful maps, figures and endless charts, but already he knows what the day will bring. Lightning will strike in places he expects. Winds will pick up, moist places will dry and flames will roar.
True Label: 3, Predicted: 0

Text: Calif. Aims to Limit Farm-Related Smog (AP) AP - Southern California's smog-fighting agency went after emissions of the bovine variety Friday, adopting the nation's first rules to reduce air pollution from dairy cow manure.
True Label: 3, Predicted: 2

Text: Loosing the War on Terrorism \\"Sven Jaschan, self-confessed aut

## 학습 2



```
labels=[
        "Answer: This is about global events, politics, and international news.",
        "Answer: This is about sports events, competitions, and athletic achievements.",
        "Answer: This article discusses business, finance, or economic topics.",
        "Answer: This is about technology, innovation, and scientific advancements."
    ]
```






In [ ]:
import numpy as np
from tqdm import tqdm


n_corrects = 0
num_samples = 50  # 테스트할 샘플 수
incorrect_samples = []  # 잘못 예측한 샘플을 저장할 리스트

for i in tqdm(range(num_samples)):
    # `testset`에서 샘플 텍스트와 라벨을 가져오기
    text = testset[i]['text']
    true_label = testset[i]['label']

    # Zero-Shot Classification 수행
    probs = zero_shot_classification(
             text,
            "Given the following article, identify which of these categories it belongs to: ",
            labels=[
                "Answer: This is about global events, politics, and international news.",
                "Answer: This is about sports events, competitions, and athletic achievements.",
                "Answer: This article discusses business, finance, or economic topics.",
                "Answer: This is about technology, innovation, and scientific advancements."
            ]
    )

    # 모델이 예측한 라벨 인덱스
    pred = np.argmax(np.array(probs))

    # 예측이 정답과 일치하는지 확인
    if pred == true_label:
        n_corrects += 1
    else:
        incorrect_samples.append((text, true_label, pred))  # 잘못 예측한 샘플 저장

print(n_corrects)

# 정확도 계산
accuracy = (n_corrects / num_samples) * 100
print(f"Accuracy: {accuracy:.2f}%")

# 잘못 예측한 샘플 출력
for sample in incorrect_samples:
    text, true_label, pred = sample
    print(f"\nText: {text}\nTrue Label: {true_label}, Predicted: {pred}")

100%|██████████| 50/50 [00:12<00:00,  4.02it/s]

21
Accuracy: 42.00%

Text: Fears for T N pension after talks Unions representing workers at Turner   Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.
True Label: 2, Predicted: 0

Text: Prediction Unit Helps Forecast Wildfires (AP) AP - It's barely dawn when Mike Fitzpatrick starts his shift with a blur of colorful maps, figures and endless charts, but already he knows what the day will bring. Lightning will strike in places he expects. Winds will pick up, moist places will dry and flames will roar.
True Label: 3, Predicted: 0

Text: Calif. Aims to Limit Farm-Related Smog (AP) AP - Southern California's smog-fighting agency went after emissions of the bovine variety Friday, adopting the nation's first rules to reduce air pollution from dairy cow manure.
True Label: 3, Predicted: 0

Text: Open Letter Against British Copyright Indoctrination in Schools The British Department for Education and Skills (DfES) recently launched a "Music Manifesto" campai

## 학습 3

       

*   "Classify the following news article into one of these categories: ",
*   labels=label_names  # 자동으로 가져온 라벨 이름 사용


        

In [ ]:
import numpy as np
from tqdm import tqdm


n_corrects = 0
num_samples = 50  # 테스트할 샘플 수
incorrect_samples = []  # 잘못 예측한 샘플을 저장할 리스트

for i in tqdm(range(num_samples)):
    # `testset`에서 샘플 텍스트와 라벨을 가져오기
    text = testset[i]['text']
    true_label = testset[i]['label']

    # Zero-Shot Classification 수행
    probs = zero_shot_classification(
        text,
        "Classify the following news article into one of these categories: ",
        labels=label_names  # 자동으로 가져온 라벨 이름 사용
    )

    # 모델이 예측한 라벨 인덱스
    pred = np.argmax(np.array(probs))

    # 예측이 정답과 일치하는지 확인
    if pred == true_label:
        n_corrects += 1
    else:
        incorrect_samples.append((text, true_label, pred))  # 잘못 예측한 샘플 저장

print(n_corrects)

# 정확도 계산
accuracy = (n_corrects / num_samples) * 100
print(f"Accuracy: {accuracy:.2f}%")

# 잘못 예측한 샘플 출력
for sample in incorrect_samples:
    text, true_label, pred = sample
    print(f"\nText: {text}\nTrue Label: {true_label}, Predicted: {pred}")

100%|██████████| 50/50 [00:11<00:00,  4.54it/s]

32
Accuracy: 64.00%

Text: Prediction Unit Helps Forecast Wildfires (AP) AP - It's barely dawn when Mike Fitzpatrick starts his shift with a blur of colorful maps, figures and endless charts, but already he knows what the day will bring. Lightning will strike in places he expects. Winds will pick up, moist places will dry and flames will roar.
True Label: 3, Predicted: 1

Text: Calif. Aims to Limit Farm-Related Smog (AP) AP - Southern California's smog-fighting agency went after emissions of the bovine variety Friday, adopting the nation's first rules to reduce air pollution from dairy cow manure.
True Label: 3, Predicted: 2

Text: Card fraud unit nets 36,000 cards In its first two years, the UK's dedicated card fraud unit, has recovered 36,000 stolen cards and 171 arrests - and estimates it saved 65m.
True Label: 3, Predicted: 2

Text: Storage, servers bruise HP earnings update Earnings per share rise compared with a year ago, but company misses analysts' expectations by a long shot.


# GPT 해설

세 가지 프롬프트는 모두 Zero-Shot Classification을 수행할 수 있지만, 각기 다른 방식으로 모델에 정보를 제공하므로 성능 차이가 있을 수 있습니다. 각 프롬프트의 장단점을 비교해 보겠습니다.


---


## 1. 첫 번째 프롬프트
",
    labels=["Answer: World.", "Answer: Sports.", "Answer: Business.", "Answer: Sci/Tech."]

- 장점
 : 간결하고 명확하여 모델이 쉽게 파악할 수 있습니다.
"Answer: "라는 형식은 모델이 직관적으로 대답을 기대하게 만들 수 있습니다.
- 단점
 : 각 라벨이 짧고 추상적이라서 모델이 잘못 이해할 가능성이 있습니다. 예를 들어, "World"라는 단어만으로는 특정한 의미를 모델에게 전달하기에 충분하지 않을 수 있습니다.
- 추천 상황
**: 모델이 이미 충분히 잘 학습된 상태에서 간단한 테스트를 하고 싶을 때.**


---



## 2. 두 번째 프롬프트 (더 설명적인 라벨)

    labels=[
        "Answer: This is about global events, politics, and international news.",
        "Answer: This is about sports events, competitions, and athletic achievements.",
        "Answer: This article discusses business, finance, or economic topics.",
        "Answer: This is about technology, innovation, and scientific advancements."
    ]

- 장점
: 라벨이 더 설명적이어서 모델이 각 카테고리의 의미를 더 잘 이해할 수 있습니다.
각 라벨이 구체적으로 어떤 주제인지 명확히 정의되어 있어, 혼동할 가능성을 줄입니다.

- 단점:
라벨이 길어지면서 처리 속도가 느려질 수 있습니다.
프롬프트가 복잡해지면 모델이 잘못된 예측을 할 때도 있으므로, 여러 번 실험이 필요할 수 있습니다.

- 추천 상황:
**모델이 라벨의 의미를 정확히 이해해야 할 때, 특히 각 카테고리가 명확히 정의되어 있는 경우에 유리합니다.**



---



## 3. 세 번째 프롬프트 (자동으로 가져온 라벨 이름)

    "Classify the following news article into one of these categories: ",
    labels=label_names  # 자동으로 가져온 라벨 이름 사용

- 장점:
코드가 더 깔끔하고 유지보수가 용이합니다. 데이터셋의 라벨 이름을 자동으로 가져와 사용하므로, 하드코딩을 피할 수 있습니다.
데이터셋과 일치하는 라벨을 사용하므로 더 일관된 예측을 기대할 수 있습니다.

- 단점:
라벨 이름이 짧아서 모델이 충분히 의미를 파악하지 못할 수 있습니다. "World"와 같은 단어만으로는 문맥을 이해하기 어려울 수 있습니다.

- 추천 상황:
**자동으로 라벨을 가져와 처리해야 할 때, 코드의 가독성과 재사용성을 높이고 싶을 때.**



---



## 결론:
정확도를 높이고 싶다면: 두 번째 프롬프트처럼 더 설명적인 라벨을 사용하는 것이 유리합니다. 이렇게 하면 모델이 각 카테고리를 더 잘 이해하고 올바른 예측을 할 가능성이 높아집니다.


간결한 테스트 또는 속도가 중요하다면: 첫 번째 프롬프트가 적합합니다. 더 짧고 직관적인 방식으로 모델의 반응을 볼 수 있습니다.


코드의 유지보수를 고려한다면: 세 번째 프롬프트를 사용하면 라벨이 변경되어도 코드 수정을 최소화할 수 있어, 다양한 데이터셋에 쉽게 적용할 수 있습니다.


개선된 성능을 위해 여러 프롬프트를 시도해보고, 그 중 가장 좋은 결과를 내는 프롬프트를 선택하는 것이 좋습니다.